# Lesson 4: Denoising Autoencoder для Fashion-MNIST

CNN-автоэнкодер для удаления гауссовского шума. Есть подбор гиперпараметров, финальное обучение, метрики и визуализации.


In [ ]:
import importlib.util
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, random_split
from torchvision import datasets, transforms
SKLEARN_AVAILABLE = importlib.util.find_spec("sklearn") is not None
if not SKLEARN_AVAILABLE:
    print("scikit-learn not found: PCA fallback will use PyTorch, t-SNE will be skipped.")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Гиперпараметры

Используется небольшой Grid Search, затем финальная модель обучается с `best_config`.


In [ ]:
BATCH_SIZE = 128
SEARCH_EPOCHS = 2
FINAL_EPOCHS = 8
GRID = [
    {"latent_dim": latent_dim, "lr": lr, "weight_decay": 1e-4}
    for latent_dim in [32, 64, 128]
    for lr in [1e-3, 5e-4]
]
NOISE_STD_MIN = 0.1
NOISE_STD_MAX = 0.4
EVAL_NOISE_STDS = [0.1, 0.2, 0.3, 0.4]
DATA_DIR = "./data"
print(f"Grid size: {len(GRID)} configs")


## Данные

Fashion-MNIST делится на `60/20/20`: обучение, валидация, тест.


In [ ]:
transform = transforms.ToTensor()
train_raw = datasets.FashionMNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)
test_raw = datasets.FashionMNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)
full_dataset = ConcatDataset([train_raw, test_raw])
total_size = len(full_dataset)
train_size = int(0.60 * total_size)
val_size = int(0.20 * total_size)
test_size = total_size - train_size - val_size
split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=split_generator,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
classes = train_raw.classes
print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")


## Шум

На обучении используется случайный `std` из `[0.1, 0.4]`; на оценке проверяются фиксированные уровни.


In [ ]:
def add_gaussian_noise(images, std=None, std_min=0.1, std_max=0.4):
    """Добавляет гауссовский шум и обрезает значения к диапазону [0, 1]."""
    if std is None:
        std = torch.empty(images.size(0), 1, 1, 1, device=images.device).uniform_(std_min, std_max)
    elif not torch.is_tensor(std):
        std = torch.full((images.size(0), 1, 1, 1), float(std), device=images.device)
    noisy = images + torch.randn_like(images) * std
    return torch.clamp(noisy, 0.0, 1.0), std
def show_noise_examples(loader, noise_stds=(0.1, 0.2, 0.3, 0.4), n=6):
    images, _ = next(iter(loader))
    images = images[:n].to(device)
    fig, axes = plt.subplots(len(noise_stds) + 1, n, figsize=(1.6 * n, 1.6 * (len(noise_stds) + 1)))
    for i in range(n):
        axes[0, i].imshow(images[i, 0].cpu(), cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_ylabel("оригинал")
    for row, std in enumerate(noise_stds, start=1):
        noisy, _ = add_gaussian_noise(images, std=std)
        for i in range(n):
            axes[row, i].imshow(noisy[i, 0].cpu(), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_ylabel(f"std={std}")
    plt.tight_layout()
    plt.show()
show_noise_examples(test_loader)


## Модель

Сверточный denoising autoencoder: encoder сжимает `1x28x28` в latent-вектор, decoder восстанавливает изображение.


In [ ]:
class DenoisingConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 7x7
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        self.to_latent = nn.Linear(128 * 7 * 7, latent_dim)
        self.from_latent = nn.Linear(latent_dim, 128 * 7 * 7)
        self.decoder_cnn = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )
    def encode(self, x):
        features = self.encoder_cnn(x)
        return self.to_latent(self.flatten(features))
    def decode(self, z):
        features = self.from_latent(z).view(-1, 128, 7, 7)
        return self.decoder_cnn(features)
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
example_model = DenoisingConvAutoencoder(latent_dim=64).to(device)
print(f"Parameters with latent_dim=64: {sum(p.numel() for p in example_model.parameters() if p.requires_grad):,}")
del example_model


## Метрики

Используются `MSE`, `PSNR`, `SSIM`.


In [ ]:
def mse_per_image(pred, target):
    return (pred - target).pow(2).flatten(1).mean(dim=1)
def psnr_from_mse(mse, max_value=1.0):
    return 10.0 * torch.log10((max_value ** 2) / (mse + 1e-8))
def simple_ssim(pred, target, max_value=1.0):
    # Упрощенная SSIM по всему изображению. Для сравнения моделей этого достаточно.
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    mu_x = pred_flat.mean(dim=1)
    mu_y = target_flat.mean(dim=1)
    var_x = pred_flat.var(dim=1, unbiased=False)
    var_y = target_flat.var(dim=1, unbiased=False)
    cov_xy = ((pred_flat - mu_x[:, None]) * (target_flat - mu_y[:, None])).mean(dim=1)
    c1 = (0.01 * max_value) ** 2
    c2 = (0.03 * max_value) ** 2
    return ((2 * mu_x * mu_y + c1) * (2 * cov_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (var_x + var_y + c2))


## Grid Search и финальное обучение

Validation используется для выбора гиперпараметров; test set используется только для итоговой оценки.


In [ ]:
criterion = nn.MSELoss()
def train_one_epoch(model, loader, optimizer, train_noise_std=None):
    model.train()
    running_loss = 0.0
    total = 0
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        if train_noise_std is None:
            noisy_images, _ = add_gaussian_noise(
                images,
                std=None,
                std_min=NOISE_STD_MIN,
                std_max=NOISE_STD_MAX,
            )
        else:
            noisy_images, _ = add_gaussian_noise(images, std=train_noise_std)
        optimizer.zero_grad(set_to_none=True)
        reconstructed = model(noisy_images)
        loss = criterion(reconstructed, images)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size
    return running_loss / total
@torch.no_grad()
def evaluate(model, loader, noise_std=0.3):
    model.eval()
    total_mse = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    total = 0
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        noisy_images, _ = add_gaussian_noise(images, std=noise_std)
        reconstructed = model(noisy_images)
        mse_values = mse_per_image(reconstructed, images)
        psnr_values = psnr_from_mse(mse_values)
        ssim_values = simple_ssim(reconstructed, images)
        batch_size = images.size(0)
        total_mse += mse_values.sum().item()
        total_psnr += psnr_values.sum().item()
        total_ssim += ssim_values.sum().item()
        total += batch_size
    return {
        "noise_std": noise_std,
        "mse": total_mse / total,
        "psnr": total_psnr / total,
        "ssim": total_ssim / total,
    }
def fit_model(config, epochs, train_loader, val_loader, train_noise_std=None, eval_noise_std=0.3, verbose=True):
    model = DenoisingConvAutoencoder(latent_dim=config["latent_dim"]).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []
    for epoch in range(1, epochs + 1):
        train_mse = train_one_epoch(model, train_loader, optimizer, train_noise_std=train_noise_std)
        val_metrics = evaluate(model, val_loader, noise_std=eval_noise_std)
        scheduler.step()
        row = {"epoch": epoch, "train_mse": train_mse, **val_metrics}
        history.append(row)
        if verbose:
            print(
                f"Epoch {epoch:02d}/{epochs} | "
                f"train MSE: {train_mse:.5f} | "
                f"val MSE: {val_metrics['mse']:.5f} | "
                f"PSNR: {val_metrics['psnr']:.2f} | "
                f"SSIM: {val_metrics['ssim']:.4f}"
            )
    return model, history
grid_results = []
best_config = GRID[0]
best_val_mse = float("inf")
start_time = time.time()
for i, config in enumerate(GRID, start=1):
    print(f"Grid {i:02d}/{len(GRID)}: {config}")
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    candidate_model, candidate_history = fit_model(
        config,
        SEARCH_EPOCHS,
        train_loader,
        val_loader,
        verbose=False,
    )
    best_epoch_row = min(candidate_history, key=lambda row: row["mse"])
    grid_row = {**config, **best_epoch_row}
    grid_results.append(grid_row)
    print(
        f"best epoch: {best_epoch_row['epoch']} | "
        f"val MSE: {best_epoch_row['mse']:.5f} | "
        f"PSNR: {best_epoch_row['psnr']:.2f} | "
        f"SSIM: {best_epoch_row['ssim']:.4f}"
    )
    if best_epoch_row["mse"] < best_val_mse:
        best_val_mse = best_epoch_row["mse"]
        best_config = config
    del candidate_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("\nGrid results sorted by validation MSE:")
for row in sorted(grid_results, key=lambda item: item["mse"]):
    print(
        f"latent={row['latent_dim']:>3}, "
        f"lr={row['lr']:.0e}, "
        f"wd={row['weight_decay']:.0e}, "
        f"epoch={row['epoch']}, "
        f"MSE={row['mse']:.5f}, "
        f"PSNR={row['psnr']:.2f}, "
        f"SSIM={row['ssim']:.4f}"
    )
print(f"\nBest config: {best_config}")
print("\nFinal run on best config")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
model, history = fit_model(best_config, FINAL_EPOCHS, train_loader, val_loader, verbose=True)
print(f"Search + final training time: {(time.time() - start_time) / 60:.1f} min")


## График финального обучения


In [ ]:
epochs = [row["epoch"] for row in history]
train_mse = [row["train_mse"] for row in history]
val_mse = [row["mse"] for row in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs, train_mse, marker="o", label="MSE на обучении")
plt.plot(epochs, val_mse, marker="o", label="MSE на валидации, std=0.3")
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Кривая обучения финальной модели")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Test set: разные уровни шума


In [ ]:
results = [evaluate(model, test_loader, noise_std=std) for std in EVAL_NOISE_STDS]
print(" noise_std |   MSE   |  PSNR  |  SSIM")
print("-----------|---------|--------|--------")
for row in results:
    print(f"   {row['noise_std']:.1f}     | {row['mse']:.5f} | {row['psnr']:.2f} | {row['ssim']:.4f}")


## Примеры восстановления

Финальная модель проверяется на одних и тех же картинках с `std=0.1`, `0.2`, `0.3`, `0.4`.


In [ ]:
@torch.no_grad()
def show_reconstructions_for_noise_levels(model, loader, noise_stds=(0.1, 0.2, 0.3, 0.4), n=8):
    model.eval()
    images, labels = next(iter(loader))
    images = images[:n].to(device)
    labels = labels[:n]
    for noise_std in noise_stds:
        torch.manual_seed(SEED + int(noise_std * 1000))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED + int(noise_std * 1000))
        noisy_images, _ = add_gaussian_noise(images, std=noise_std)
        reconstructed = model(noisy_images)
        rows = [
            ("оригинал", images),
            (f"шум std={noise_std}", noisy_images),
            ("восстановлено", reconstructed),
        ]
        fig, axes = plt.subplots(len(rows), n, figsize=(1.8 * n, 5.0))
        fig.suptitle(f"Одни и те же тестовые изображения, шум std={noise_std}", y=0.99)
        for row_idx, (row_name, batch) in enumerate(rows):
            for col_idx in range(n):
                axes[row_idx, col_idx].imshow(batch[col_idx, 0].cpu(), cmap="gray")
                axes[row_idx, col_idx].axis("off")
                if row_idx == 0:
                    axes[row_idx, col_idx].set_title(classes[labels[col_idx]], fontsize=8)
                if col_idx == 0:
                    axes[row_idx, col_idx].set_ylabel(row_name, rotation=0, ha="right", va="center", labelpad=58)
        plt.tight_layout(rect=(0.10, 0.00, 1.00, 0.96))
        plt.show()
show_reconstructions_for_noise_levels(model, test_loader, noise_stds=EVAL_NOISE_STDS, n=8)


## PCA latent space

Цвет точки соответствует классу Fashion-MNIST.


In [ ]:
@torch.no_grad()
def collect_latents(model, loader, max_samples=2000, noise_std=0.3):
    model.eval()
    latents = []
    labels_all = []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        noisy_images, _ = add_gaussian_noise(images, std=noise_std)
        z = model.encode(noisy_images)
        latents.append(z.cpu())
        labels_all.append(labels)
        if sum(x.size(0) for x in latents) >= max_samples:
            break
    latents = torch.cat(latents, dim=0)[:max_samples].numpy()
    labels_all = torch.cat(labels_all, dim=0)[:max_samples].numpy()
    return latents, labels_all
latents, latent_labels = collect_latents(model, test_loader, max_samples=2000, noise_std=0.3)
if SKLEARN_AVAILABLE:
    from sklearn.decomposition import PCA
    pca_points = PCA(n_components=2, random_state=SEED).fit_transform(latents)
else:
    latents_tensor = torch.tensor(latents, dtype=torch.float32)
    centered = latents_tensor - latents_tensor.mean(dim=0, keepdim=True)
    _, _, components = torch.pca_lowrank(centered, q=2)
    pca_points = (centered @ components[:, :2]).numpy()
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    pca_points[:, 0],
    pca_points[:, 1],
    c=latent_labels,
    cmap="tab10",
    s=10,
    alpha=0.75,
)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.ax.set_yticklabels(classes)
plt.title("Latent space PCA, тестовые изображения с шумом std=0.3")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.2)
plt.show()


## t-SNE

По умолчанию выключено, потому что работает дольше PCA.


In [ ]:
RUN_TSNE = False
if RUN_TSNE and not SKLEARN_AVAILABLE:
    print("t-SNE requires scikit-learn. Install it or use the PCA visualization above.")
elif RUN_TSNE:
    from sklearn.manifold import TSNE
    tsne_points = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate="auto",
        init="pca",
        random_state=SEED,
    ).fit_transform(latents)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        tsne_points[:, 0],
        tsne_points[:, 1],
        c=latent_labels,
        cmap="tab10",
        s=10,
        alpha=0.75,
    )
    cbar = plt.colorbar(scatter, ticks=range(10))
    cbar.ax.set_yticklabels(classes)
    plt.title("Latent space t-SNE, тестовые изображения с шумом std=0.3")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.grid(alpha=0.2)
    plt.show()
else:
    print("t-SNE skipped. Set RUN_TSNE = True to run it.")


## Вывод

В ноутбуке обучается одна CNN-модель на смешанном гауссовском шуме `std=0.1..0.4`. Гиперпараметры выбираются небольшим Grid Search, затем финальная модель оценивается на test set при `std=0.1`, `0.2`, `0.3`, `0.4`.
